# 02 - Configure the Connected Model

Use **Python (byom01-capability-testing)** with the working directory set to `notebooks/`.

Complete [01_deploy_infra.ipynb](./01_deploy_infra.ipynb) first. This notebook reads the generated deployment settings from `env/.env`.

Configure gateway policies, scoped permissions, a Foundry project connection, and Application Insights with Log Analytics through Bicep. This modifies deployed resources and creates paid telemetry resources. The signed-in user needs deployment/connection permissions and role-assignment permission on the backend and telemetry component.

**[PREVIEW] Upstream Responses routing must be available in this Foundry project.** Creating a connection does not enable the feature. See [the gateway guide](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/ai-gateway).

The consuming project authenticates to APIM with its identity; APIM uses its own identity to invoke GPT-5.4. No keys, prompts, or response bodies are logged.

To allow your user to call APIM directly in [05_direct_api.ipynb](./05_direct_api.ipynb), set `APIM_ALLOWED_USER_OBJECT_ID` in `env/.env` before running Configure. Use your object ID in the deployment tenant (`az ad signed-in-user show --query id -o tsv`). Empty keeps APIM access limited to the consuming project identity; it does not allow direct user calls.

## Load Deployment Settings

In [ ]:
import json
import subprocess
import sys
from pathlib import Path
from uuid import UUID

from azure.identity import AzureCliCredential
from azure.mgmt.resource import ResourceManagementClient
from dotenv import dotenv_values, set_key

demo_root = Path("..").resolve()
env_path = demo_root / "env" / ".env"
base_settings = dotenv_values(env_path)
allowed_user_object_id = (base_settings.get("APIM_ALLOWED_USER_OBJECT_ID") or "").strip()
if allowed_user_object_id:
    allowed_user_object_id = str(UUID(allowed_user_object_id))
required_settings = [
    "AZURE_SUBSCRIPTION_ID", "AZURE_RESOURCE_GROUP", "AZURE_LOCATION",
    "AZURE_ACCOUNT_NAME", "AZURE_AI_PROJECT_NAME", "AZURE_AI_PROJECT_ID",
    "AZURE_AI_PROJECT_ENDPOINT", "AZURE_APIM_NAME", "AZURE_BACKEND_RESOURCE_GROUP",
    "AZURE_BACKEND_ACCOUNT_NAME", "AZURE_BACKEND_OPENAI_ENDPOINT",
    "AZURE_BACKEND_DEPLOYMENT_NAME", "AZURE_BACKEND_MODEL_VERSION",
]
settings: dict[str, str] = {}
for name in required_settings:
    value = base_settings[name]
    if not value:
        raise ValueError(f"Missing generated setting {name}; complete deployment first.")
    settings[name] = value

credential = AzureCliCredential(subscription=settings["AZURE_SUBSCRIPTION_ID"])
resource_client = ResourceManagementClient(credential, settings["AZURE_SUBSCRIPTION_ID"])
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))
from app.notebook_helpers import resolve_az_cli

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found. Run the setup notebook first.")
gateway_path = demo_root / "env" / "gateway.json"
print(f"Subscription: {settings['AZURE_SUBSCRIPTION_ID']}")
print(f"Consumer: {settings['AZURE_RESOURCE_GROUP']}; backend: {settings['AZURE_BACKEND_RESOURCE_GROUP']}")

## Compile Gateway Bicep

The single documented BCP036 accommodation covers Bicep's stale `ProjectManagedIdentity` type. All other compiler diagnostics remain active.

In [ ]:
build_result = subprocess.run(
    [az_cmd, "bicep", "build", "--file", str(demo_root / "infra" / "gateway.bicep"), "--stdout"],
    capture_output=True, text=True,
)
if build_result.returncode != 0:
    raise RuntimeError(
        f"az bicep build failed (exit {build_result.returncode})\n"
        f"stderr: {build_result.stderr.strip()}\nstdout: {build_result.stdout.strip()}"
    )
gateway_template = json.loads(build_result.stdout)
print("Gateway Bicep compiled")

## Deploy Gateway Configuration

Wait for completion. The backend-scoped module creates only an inference role assignment, not another model. Unchanged settings reuse the same resources. RBAC propagation may take time before the Test notebook can invoke the backend.

Do not run the write step after a failed deployment; a previous gateway file describes only its previous successful configuration.

In [ ]:
gateway_outputs = None
gateway_settings = None
parameters = {
    "foundryAccountName": settings["AZURE_ACCOUNT_NAME"],
    "projectName": settings["AZURE_AI_PROJECT_NAME"],
    "apimName": settings["AZURE_APIM_NAME"],
    "backendResourceGroup": settings["AZURE_BACKEND_RESOURCE_GROUP"],
    "backendAccountName": settings["AZURE_BACKEND_ACCOUNT_NAME"],
    "backendOpenAIEndpoint": settings["AZURE_BACKEND_OPENAI_ENDPOINT"],
    "backendDeploymentName": settings["AZURE_BACKEND_DEPLOYMENT_NAME"],
    "backendModelVersion": settings["AZURE_BACKEND_MODEL_VERSION"],
    "location": settings["AZURE_LOCATION"],
    "allowedUserObjectId": allowed_user_object_id,
}
deployment = resource_client.deployments.begin_create_or_update(
    settings["AZURE_RESOURCE_GROUP"], "gateway-byom01",
    {"properties": {
        "mode": "Incremental", "template": gateway_template,
        "parameters": {name: {"value": value} for name, value in parameters.items()},
    }},
).result()
gateway_outputs = deployment.properties.outputs
print("Gateway deployment completed")

## Read Registered Connection

Verify the saved project connection before producing the Test notebook's settings. This is configuration verification, not inference evidence.

In [ ]:
configured = {name: output["value"] for name, output in gateway_outputs.items()}
connection = resource_client.resources.get_by_id(configured["connectionId"], "2026-09-01")
properties = connection.properties
assert properties["category"] == "ApiManagement", "Unexpected connection category"
assert properties["authType"] == "ProjectManagedIdentity", "Unexpected connection authentication"
assert properties["audience"] == "https://cognitiveservices.azure.com", "Unexpected token audience"
assert properties["isSharedToAll"] is False, "Connection must remain project scoped"
assert properties["target"] == configured["gatewayUrl"], "Unexpected gateway target"
metadata = properties["metadata"]
assert metadata["deploymentInPath"] == "false", "Deployment must be in the request body"
assert metadata["upstreamSurface"] == "responses", "Responses routing was not saved"
assert json.loads(metadata["customHeaders"]) == {}, "Unexpected custom headers"
assert json.loads(metadata["models"]) == [{
    "name": settings["AZURE_BACKEND_DEPLOYMENT_NAME"],
    "properties": {"model": {
        "name": "gpt-5.4", "version": settings["AZURE_BACKEND_MODEL_VERSION"], "format": "OpenAI",
    }},
}], "Unexpected registered model"
gateway_settings = {
    **configured,
    "subscriptionId": settings["AZURE_SUBSCRIPTION_ID"],
    "projectId": settings["AZURE_AI_PROJECT_ID"],
    "projectEndpoint": settings["AZURE_AI_PROJECT_ENDPOINT"],
}
print(f"Connection: {configured['connectionId']}")
print(f"Callable model: {configured['modelReference']}")
print(f"Gateway: {configured['gatewayUrl']}/responses")

## Write Gateway Settings

Save the test handoff and generate direct APIM and Foundry settings in `env/.env` only after successful deployment/readback. Use the consuming project endpoint for Foundry, with the registered connected-model reference and Foundry token scope. Update only these URL/model/scope fields, preserving keys, unrelated fields, and comments. An external endpoint user can instead supply equivalent file settings without deploying this repository. Generated settings do not prove direct Foundry Responses support.

In [ ]:
assert gateway_settings is not None, "Read and verify the connection before writing settings"
gateway_path.write_text(json.dumps(gateway_settings, indent=2) + "\n", encoding="utf-8")
direct_settings = {
    "APIM_BASE_URL": gateway_settings["gatewayUrl"].rstrip("/") + "/",
    "APIM_MODEL_NAME": gateway_settings["deploymentName"],
    "APIM_TOKEN_SCOPE": gateway_settings["directTokenScope"],
    "FOUNDRY_BASE_URL": gateway_settings["projectEndpoint"].rstrip("/") + "/openai/v1/",
    "FOUNDRY_MODEL_NAME": gateway_settings["modelReference"],
    "FOUNDRY_TOKEN_SCOPE": "https://ai.azure.com/.default",
}
for name, value in direct_settings.items():
    set_key(env_path, name, value)
print(f"Wrote {gateway_path} and direct APIM/Foundry settings in {env_path}")
print("Direct APIM Entra calls require the selected credential's user to match APIM_ALLOWED_USER_OBJECT_ID.")

Continue with [03_test.ipynb](03_test.ipynb). Its signed-in user needs **Foundry User** on the project and **Log Analytics Reader** (or equivalent existing access) on the generated workspace. Configuration creates only APIM's service permissions, not broader user permissions.

Full-demo destruction removes the connection and telemetry with the consumer group, and backend-scoped access with the backend account. Follow the [README](../README.md#cleanup); never use full-demo deletion as cleanup for tests against pre-existing resources.